In [ ]:
import sys
sys.path.insert(0, '/home/projects/nyosef/zvise/PixelGen/PixelGen')

import os
import numpy as np
import pandas as pd
import scanpy as sc
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from pixelator import read_pna
import igraph as ig
import leidenalg
import networkx as nx
from scipy import linalg as la
from sklearn.cluster import KMeans
from sklearn.metrics import adjusted_rand_score, normalized_mutual_info_score
from collections import defaultdict

# --- Global figure style ---
sns.set_style("whitegrid")
sc.settings.set_figure_params(dpi=120, frameon=False, fontsize=12)
plt.rcParams.update({
    "figure.figsize": (8, 5),
    "axes.titlesize": 13,
    "axes.labelsize": 12,
    "xtick.labelsize": 10,
    "ytick.labelsize": 10,
    "legend.fontsize": 10,
    "font.family": "sans-serif",
})

RESULTS_DIR = Path("/home/projects/nyosef/zvise/PixelGen/PixelGen/New_Data/results")
CACHE_DIR = Path("/home/projects/nyosef/zvise/PixelGen/PixelGen/New_Data/cache")

# Color palettes
COND_PALETTE = {"Mock": "#4c72b0", "Blinatumomab": "#dd8452"}
TIME_PALETTE = {"6h": "#55a868", "48h": "#c44e52"}
SYSTEM_PALETTE = {
    "healthy B + healthy T": "#4878d0",
    "NALM-6 + healthy T": "#ee854a",
    "patient B + patient T": "#6acc65",
    "NALM-6 + patient T": "#d65f5f",
}

# Sample metadata
sample_meta = {
    "S001": {"time": "6h",  "condition": "Mock",           "target": "healthy B",  "tcells": "healthy T"},
    "S002": {"time": "6h",  "condition": "Blinatumomab",   "target": "healthy B",  "tcells": "healthy T"},
    "S003": {"time": "48h", "condition": "Mock",           "target": "healthy B",  "tcells": "healthy T"},
    "S004": {"time": "48h", "condition": "Blinatumomab",   "target": "healthy B",  "tcells": "healthy T"},
    "S005": {"time": "6h",  "condition": "Mock",           "target": "NALM-6",     "tcells": "healthy T"},
    "S006": {"time": "6h",  "condition": "Blinatumomab",   "target": "NALM-6",     "tcells": "healthy T"},
    "S007": {"time": "48h", "condition": "Mock",           "target": "NALM-6",     "tcells": "healthy T"},
    "S008": {"time": "48h", "condition": "Blinatumomab",   "target": "NALM-6",     "tcells": "healthy T"},
    "S009": {"time": "6h",  "condition": "Mock",           "target": "patient B",  "tcells": "patient T"},
    "S010": {"time": "6h",  "condition": "Blinatumomab",   "target": "patient B",  "tcells": "patient T"},
    "S011": {"time": "48h", "condition": "Mock",           "target": "patient B",  "tcells": "patient T"},
    "S012": {"time": "48h", "condition": "Blinatumomab",   "target": "patient B",  "tcells": "patient T"},
    "S013": {"time": "6h",  "condition": "Mock",           "target": "NALM-6",     "tcells": "patient T"},
    "S014": {"time": "6h",  "condition": "Blinatumomab",   "target": "NALM-6",     "tcells": "patient T"},
    "S016": {"time": "48h", "condition": "Blinatumomab",   "target": "NALM-6",     "tcells": "patient T"},
}

# Isotype controls to exclude from community detection
ISOTYPE_CONTROLS = {"mIgG1", "mIgG2a", "mIgG2b"}

In [ ]:
# Load annotated adata and filter to CD8 T cells
ANNOTATED_CACHE = CACHE_DIR / 'adata_cytovi_annotated_compat.h5ad'
adata = sc.read_h5ad(ANNOTATED_CACHE)

# Filter to CD8 only
adata = adata[adata.obs["cell_type"] == "CD8"].copy()

# Standardise column names
if "system" not in adata.obs.columns and "cell_system" in adata.obs.columns:
    adata.obs["system"] = adata.obs["cell_system"]
if "cond_time" not in adata.obs.columns:
    adata.obs["cond_time"] = adata.obs["condition"].astype(str) + "_" + adata.obs["time"].astype(str)

cd8_cells = set(adata.obs.index)

print(f"adata (CD8 only): {adata.shape}")
print(adata.obs.groupby(["system", "cond_time"]).size().unstack(fill_value=0))

In [ ]:
# =============================================================================
# DATA PREPARATION: Load proximity data and filter to CD8 cells
# =============================================================================

# Load proximity data
cache_path = CACHE_DIR / "spatial_all_samples.parquet"
if cache_path.exists():
    print(f"Loading cached proximity data from {cache_path}")
    proximity = pd.read_parquet(cache_path)
else:
    raise FileNotFoundError(f"Proximity cache not found: {cache_path}")

print(f"Proximity shape (all cells): {proximity.shape}")

# Filter proximity to CD8 cells only
proximity = proximity[proximity["component"].isin(cd8_cells)].copy()
print(f"Proximity shape (CD8 only): {proximity.shape}")
print(f"Unique CD8 cells in proximity: {proximity['component'].nunique()}")

# Marker Colocalization Graph — Single Cell Example

Visualize the **marker–marker colocalization graph** for one representative cell.  
Nodes = protein markers, edges = significant colocalization (join-count z-score).  
Node color encodes the **Leiden CPM community** assignment.

In [ ]:
# =============================================================================
# Force-directed graph of marker colocalization for a single CD8 cell
# =============================================================================

# --- Load pre-computed Leiden CPM community assignments (resolution=0.1) ---
leiden_df = pd.read_parquet(CACHE_DIR / "percell_leiden_cpm.parquet")
marker_cols = [c for c in leiden_df.columns if c != "condition"]

# Filter to CD8 cells
leiden_df = leiden_df.loc[leiden_df.index.intersection(cd8_cells)]
print(f"Leiden CPM results for CD8 cells: {len(leiden_df)}")

# Pick a representative CD8 cell (most edges in proximity data = richest graph)
cell_edge_counts = proximity.groupby("component").size().sort_values(ascending=False)
EXAMPLE_CELL = cell_edge_counts.index[0]
print(f"Example CD8 cell: {EXAMPLE_CELL} ({cell_edge_counts.iloc[0]} proximity edges)")

cell_communities = leiden_df.loc[EXAMPLE_CELL, marker_cols]

from collections import Counter
comm_sizes = Counter(cell_communities.values)
print(f"Leiden CPM (res=0.1): {len(comm_sizes)} communities")
print(f"Community sizes: {sorted(comm_sizes.values(), reverse=True)}")

# --- Build colocalization graph from proximity data ---
cell_prox = proximity[proximity["component"] == EXAMPLE_CELL].copy()

# Keep edges with |z| > 2 (significant positive OR negative colocalization)
cell_prox = cell_prox[cell_prox["join_count_z"].abs() > 2.0]

G = nx.Graph()
G.add_nodes_from(marker_cols)
for _, row in cell_prox.iterrows():
    m1, m2 = row["marker_1"], row["marker_2"]
    if m1 in marker_cols and m2 in marker_cols and m1 != m2:
        G.add_edge(m1, m2, weight=row["join_count_z"])

isolates = list(nx.isolates(G))
G.remove_nodes_from(isolates)
print(f"Graph: {G.number_of_nodes()} nodes, {G.number_of_edges()} edges  "
      f"({len(isolates)} isolated markers removed)")

# --- Community colors ---
unique_comms = sorted(cell_communities[list(G.nodes)].unique())
comm_counts = Counter(cell_communities[list(G.nodes)].values)
large_comms = [c for c, cnt in comm_counts.items() if cnt > 2]
comm_remap = {c: c if c in large_comms else -1 for c in unique_comms}
node_comms = [comm_remap[cell_communities[n]] for n in G.nodes]

display_comms = sorted(set(node_comms))
n_colors = len(display_comms)
palette = sns.color_palette("Set2", n_colors) if n_colors <= 8 else sns.color_palette("tab20", n_colors)
comm_to_color = {c: palette[i] for i, c in enumerate(display_comms)}
node_colors = [comm_to_color[c] for c in node_comms]

# --- Force-directed layout ---
pos = nx.spring_layout(G, k=1.2 / np.sqrt(G.number_of_nodes()),
                       iterations=150, seed=42, weight="weight")

# --- Plot ---
fig, ax = plt.subplots(figsize=(14, 11))

# Edge colors (blue=positive, red=negative) and widths proportional to |weight|
edge_weights = np.array([G[u][v]["weight"] for u, v in G.edges()])
abs_weights = np.abs(edge_weights)
max_abs = abs_weights.max()
edge_widths = 0.3 + 2.5 * (abs_weights / max_abs)
edge_colors_list = [("#4878d0" if w > 0 else "#c44e52") for w in edge_weights]
edge_alphas = 0.05 + 0.25 * (abs_weights / max_abs)

for idx, (u, v) in enumerate(G.edges()):
    nx.draw_networkx_edges(G, pos, ax=ax, edgelist=[(u, v)],
                           width=edge_widths[idx], edge_color=[edge_colors_list[idx]],
                           alpha=float(edge_alphas[idx]))

# Node sizes by weighted degree
node_sizes = []
for n_node in G.nodes:
    degree = G.degree(n_node, weight="weight")
    node_sizes.append(max(40, min(300, degree * 0.5)))

nx.draw_networkx_nodes(G, pos, ax=ax, node_color=node_colors,
                       node_size=node_sizes, edgecolors="white", linewidths=0.5)

# Label top 5 most connected nodes per large community
labeled_nodes = set()
for comm in large_comms:
    comm_nodes = [nd for nd in G.nodes if comm_remap[cell_communities[nd]] == comm]
    top = sorted(comm_nodes, key=lambda nd: G.degree(nd, weight="weight"), reverse=True)[:5]
    labeled_nodes.update(top)

nx.draw_networkx_labels(G, pos, ax=ax,
                        labels={nd: nd for nd in labeled_nodes},
                        font_size=7, font_weight="bold")

# Legend
from matplotlib.lines import Line2D
legend_handles = []
for comm in display_comms:
    n_members = sum(1 for c in node_comms if c == comm)
    label = f"Community {comm} (n={n_members})" if comm != -1 else f"Other (n={n_members})"
    legend_handles.append(Line2D([0], [0], marker='o', color='w',
                                 markerfacecolor=comm_to_color[comm],
                                 markersize=10, label=label))
legend_handles.append(Line2D([0], [0], color="#4878d0", linewidth=2, label="Positive coloc."))
legend_handles.append(Line2D([0], [0], color="#c44e52", linewidth=2, label="Negative coloc."))

ax.legend(handles=legend_handles, loc="upper left", frameon=True,
          fontsize=9, title="Leiden CPM Community", title_fontsize=10)

ax.set_title(f"Marker Colocalization Graph — CD8 Cell {EXAMPLE_CELL}\n"
             f"(Leiden CPM res=0.1, edges: |z| > 2, spring layout)",
             fontsize=14, fontweight="bold")
ax.axis("off")
plt.tight_layout()
plt.show()

In [ ]:
# =============================================================================
# Load per-cell community detection results (from LSF job) — filter to CD8
# =============================================================================

ALGO_FILES = {
    "Leiden CPM": CACHE_DIR / "percell_leiden_cpm.parquet",
}

percell = {}
for algo_name, path in ALGO_FILES.items():
    if path.exists():
        df = pd.read_parquet(path)
        # Filter to CD8 cells
        cd8_idx = df.index.intersection(cd8_cells)
        percell[algo_name] = df.loc[cd8_idx]
        print(f"{algo_name}: {df.shape} → CD8: {percell[algo_name].shape}")
    else:
        print(f"WARNING: {path} not found — run the LSF job first")

# Get marker list from first available result
all_markers = [c for c in next(iter(percell.values())).columns if c != "condition"]
print(f"\nMarkers: {len(all_markers)}")

In [ ]:
# =============================================================================
# Per-cell features: n_communities, entropy, std of sizes — all 3 algorithms
# =============================================================================
from scipy.stats import entropy, mannwhitneyu

def compute_cell_features(df, marker_list):
    """Per-cell community structure features."""
    mat = df[marker_list].values
    feats = []
    for i in range(len(mat)):
        unique, counts = np.unique(mat[i], return_counts=True)
        feats.append({
            "n_communities": len(unique),
            "community_entropy": entropy(counts),
            "community_size_std": np.std(counts),
        })
    return pd.DataFrame(feats, index=df.index)

# Compute for each algorithm
cell_feats = {}
for algo_name, df in percell.items():
    cell_feats[algo_name] = compute_cell_features(df, all_markers)
    print(f"{algo_name}: computed features for {len(df)} cells")

# Build combined feature DataFrame with metadata
feat_dfs = []
for algo_name, feat_df in cell_feats.items():
    renamed = feat_df.copy()
    renamed.columns = [f"{c}__{algo_name}" for c in renamed.columns]
    feat_dfs.append(renamed)

cell_feat_df = pd.concat(feat_dfs, axis=1)
common_idx = cell_feat_df.index.intersection(adata.obs.index)
cell_feat_df = cell_feat_df.loc[common_idx]
cell_feat_df["condition"] = adata.obs.loc[common_idx, "condition"].values
cell_feat_df["time"] = adata.obs.loc[common_idx, "time"].values
cell_feat_df["system"] = adata.obs.loc[common_idx, "system"].values
cell_feat_df["cond_time"] = adata.obs.loc[common_idx, "cond_time"].values
cell_feat_df["target"] = adata.obs.loc[common_idx, "target"].values
cell_feat_df["tcells"] = adata.obs.loc[common_idx, "tcells"].values

print(f"\nCombined feature table: {cell_feat_df.shape}")
print(cell_feat_df.groupby(["system", "time", "condition"]).size().unstack(fill_value=0))

In [ ]:
# =============================================================================
# Compare per-cell features across conditions — FIGURES
# Hierarchy: system → time (6h vs 48h) → treatment (Mock vs Blina)
# =============================================================================

metrics = ["n_communities", "community_entropy", "community_size_std"]
metric_labels = {"n_communities": "Number of communities",
                 "community_entropy": "Community entropy (Shannon)",
                 "community_size_std": "Community size std"}
algo_names = list(percell.keys())

def mwu(a, b):
    if len(a) < 3 or len(b) < 3:
        return np.nan, np.nan
    stat, pval = mannwhitneyu(a, b, alternative="two-sided")
    rbc = 1 - (2 * stat) / (len(a) * len(b))
    return pval, rbc

def significance_star(p):
    if p < 0.001: return "***"
    if p < 0.01: return "**"
    if p < 0.05: return "*"
    return "ns"

# --- Figure 1: Violin plots per system, split by cond_time ---
# One figure per algorithm, rows=systems, cols=metrics
systems = sorted(cell_feat_df["system"].unique())
ct_order = ["Mock_6h", "Blinatumomab_6h", "Mock_48h", "Blinatumomab_48h"]
ct_palette = {"Mock_6h": "#7fbf7b", "Blinatumomab_6h": "#d6604d",
              "Mock_48h": "#1b7837", "Blinatumomab_48h": "#b2182b"}

for algo in algo_names:
    fig, axes = plt.subplots(len(systems), len(metrics), figsize=(5*len(metrics), 4*len(systems)))
    for row, system in enumerate(systems):
        sys_df = cell_feat_df[cell_feat_df["system"] == system].copy()
        for col_idx, metric in enumerate(metrics):
            ax = axes[row, col_idx]
            col = f"{metric}__{algo}"
            # Only keep cond_times that exist
            existing_ct = [ct for ct in ct_order if ct in sys_df["cond_time"].values]
            sns.violinplot(data=sys_df, x="cond_time", y=col, order=existing_ct,
                           palette=ct_palette, inner="box", cut=0, ax=ax, linewidth=0.8)
            ax.set_xlabel("")
            ax.set_ylabel(metric_labels[metric] if col_idx == 0 or row == 0 else "")
            if row == 0:
                ax.set_title(metric_labels[metric], fontsize=11)
            if col_idx == 0:
                ax.annotate(system, xy=(-0.45, 0.5), xycoords="axes fraction",
                            fontsize=10, fontweight="bold", rotation=90, va="center", ha="center")
            ax.tick_params(axis="x", rotation=45, labelsize=8)
            
            # Add p-value annotations: Mock vs Blina within each time
            for t_idx, t in enumerate(["6h", "48h"]):
                mock_ct = f"Mock_{t}"
                blina_ct = f"Blinatumomab_{t}"
                if mock_ct in existing_ct and blina_ct in existing_ct:
                    vals_m = sys_df[sys_df["cond_time"] == mock_ct][col].dropna()
                    vals_b = sys_df[sys_df["cond_time"] == blina_ct][col].dropna()
                    p, _ = mwu(vals_m, vals_b)
                    if not np.isnan(p):
                        star = significance_star(p)
                        x1 = existing_ct.index(mock_ct)
                        x2 = existing_ct.index(blina_ct)
                        ymax = sys_df[col].quantile(0.98)
                        h = (sys_df[col].max() - sys_df[col].min()) * 0.05
                        y = ymax + h * (1 + t_idx * 2.5)
                        ax.plot([x1, x1, x2, x2], [y, y+h, y+h, y], lw=1, c="k")
                        ax.text((x1+x2)/2, y+h, star, ha="center", va="bottom", fontsize=9)

    fig.suptitle(f"{algo}: per-cell community features by condition", fontsize=13, y=1.01)
    plt.tight_layout()
    plt.show()

# --- Figure 2: 6h vs 48h comparison per system (boxplot, grouped) ---
for algo in algo_names:
    fig, axes = plt.subplots(1, len(metrics), figsize=(5*len(metrics), 5))
    for col_idx, metric in enumerate(metrics):
        ax = axes[col_idx]
        col = f"{metric}__{algo}"
        plot_df = cell_feat_df[["system", "time", col]].copy()
        plot_df = plot_df.rename(columns={col: "value"})
        sns.boxplot(data=plot_df, x="system", y="value", hue="time",
                    palette=TIME_PALETTE, ax=ax, fliersize=1, linewidth=0.8)
        ax.set_title(metric_labels[metric])
        ax.set_xlabel("")
        ax.tick_params(axis="x", rotation=30, labelsize=8)
        if col_idx > 0:
            ax.get_legend().remove()
        else:
            ax.legend(title="Time", fontsize=9)
    fig.suptitle(f"{algo}: 6h vs 48h per system", fontsize=13, y=1.01)
    plt.tight_layout()
    plt.show()

# --- Collect all stats into a DataFrame ---
stats_rows = []
for system in systems:
    sys_df = cell_feat_df[cell_feat_df["system"] == system]
    for algo in algo_names:
        for metric in metrics:
            col = f"{metric}__{algo}"
            # 6h vs 48h
            v6 = sys_df[sys_df["time"] == "6h"][col].dropna()
            v48 = sys_df[sys_df["time"] == "48h"][col].dropna()
            p, r = mwu(v6, v48)
            stats_rows.append({"system": system, "algo": algo, "metric": metric,
                               "comparison": "6h vs 48h", "median_A": v6.median(),
                               "median_B": v48.median(), "p": p, "rbc": r})
            # Mock vs Blina per time
            for t in ["6h", "48h"]:
                t_df = sys_df[sys_df["time"] == t]
                vm = t_df[t_df["condition"] == "Mock"][col].dropna()
                vb = t_df[t_df["condition"] == "Blinatumomab"][col].dropna()
                p, r = mwu(vm, vb)
                stats_rows.append({"system": system, "algo": algo, "metric": metric,
                                   "comparison": f"{t} Mock vs Blina", "median_A": vm.median(),
                                   "median_B": vb.median(), "p": p, "rbc": r})

stats_df = pd.DataFrame(stats_rows)
sig = stats_df[stats_df["p"] < 0.05].sort_values("p")
print(f"Significant comparisons (p<0.05): {len(sig)}/{len(stats_df)}")
print(sig.to_string(index=False))

In [ ]:
# =============================================================================
# Co-assignment analysis: NALM-6 + healthy T, Blinatumomab — 6h vs 48h
# =============================================================================

def co_assignment_by_group(df, marker_list, group_col):
    """Compute marker×marker co-assignment probability per group."""
    groups = df[group_col].dropna().unique()
    results = {}
    for grp in groups:
        mat = df[df[group_col] == grp][marker_list].values
        n_cells = len(mat)
        if n_cells == 0:
            continue
        n_m = len(marker_list)
        co = np.zeros((n_m, n_m))
        for row in mat:
            same = (row[:, None] == row[None, :]).astype(np.float32)
            co += same
        co /= n_cells
        results[grp] = pd.DataFrame(co, index=marker_list, columns=marker_list)
    return results

def log2_fc_matrix(p_num, p_den, eps=1e-6):
    """Log2 fold-change of co-assignment probabilities."""
    p_num = np.clip(p_num, eps, 1.0)
    p_den = np.clip(p_den, eps, 1.0)
    return np.log2(p_num / p_den)

# --- Subset: NALM-6 + healthy T, Blinatumomab only ---
algo_repr = algo_names[0]
df_repr = percell[algo_repr].loc[percell[algo_repr].index.intersection(adata.obs.index)].copy()

focus_mask = (df_repr["system"] == "NALM-6 + healthy T") & (df_repr["condition"] == "Blinatumomab")
df_focus = df_repr.loc[focus_mask].copy()
print(f"NALM-6 + healthy T, Blinatumomab CD8 cells:")
print(f"  6h:  {(df_focus['time'] == '6h').sum()} cells")
print(f"  48h: {(df_focus['time'] == '48h').sum()} cells")

co_by_time = co_assignment_by_group(df_focus, all_markers, "time")

# --- Figure 1 & 2: Co-assignment heatmaps for 6h and 48h (clustered) ---
from scipy.cluster.hierarchy import linkage
from scipy.spatial.distance import squareform

# Compute shared clustering from the mean co-assignment
mean_co = (co_by_time["6h"].values + co_by_time["48h"].values) / 2
dist = 1 - mean_co
np.fill_diagonal(dist, 0)
condensed = squareform(dist, checks=False)
row_linkage = linkage(condensed, method="ward")

n_markers = len(all_markers)
fig_size = max(10, n_markers * 0.15)

for time_label in ["6h", "48h"]:
    co_mat = co_by_time[time_label]
    n_cells = (df_focus["time"] == time_label).sum()
    g = sns.clustermap(co_mat, cmap="YlOrRd", vmin=0, vmax=1,
                       row_linkage=row_linkage, col_linkage=row_linkage,
                       figsize=(fig_size, fig_size), linewidths=0,
                       xticklabels=True, yticklabels=True,
                       cbar_kws={"shrink": 0.4, "label": "Co-assignment probability"},
                       dendrogram_ratio=0.08, cbar_pos=(0.02, 0.82, 0.03, 0.15))
    g.ax_heatmap.tick_params(axis="both", labelsize=max(4, min(7, 180 // n_markers)))
    g.fig.suptitle(f"NALM-6 + healthy T | Blinatumomab {time_label} (n={n_cells} CD8 cells)\n"
                   f"Marker co-assignment probability ({algo_repr})",
                   fontsize=12, y=1.01)
    plt.show()

# --- Figure 3: Log2 fold-change heatmap (48h / 6h) ---
lfc_mat = pd.DataFrame(
    log2_fc_matrix(co_by_time["48h"].values, co_by_time["6h"].values),
    index=all_markers, columns=all_markers
)
np.fill_diagonal(lfc_mat.values, 0)  # diagonal is always 1/1 = 0

vmax_lfc = np.percentile(np.abs(lfc_mat.values[np.triu_indices_from(lfc_mat.values, k=1)]), 99)

g = sns.clustermap(lfc_mat, cmap="RdBu_r", center=0, vmin=-vmax_lfc, vmax=vmax_lfc,
                   row_linkage=row_linkage, col_linkage=row_linkage,
                   figsize=(fig_size, fig_size), linewidths=0,
                   xticklabels=True, yticklabels=True,
                   cbar_kws={"shrink": 0.4, "label": "log2 FC (48h / 6h)"},
                   dendrogram_ratio=0.08, cbar_pos=(0.02, 0.82, 0.03, 0.15))
g.ax_heatmap.tick_params(axis="both", labelsize=max(4, min(7, 180 // n_markers)))
g.fig.suptitle(f"NALM-6 + healthy T | Blinatumomab: co-assignment log2 FC (48h / 6h)",
               fontsize=12, y=1.01)
plt.show()

# --- Figure 4: Top 25 UP and top 25 DOWN co-assignment pairs ---
upper = np.triu_indices_from(lfc_mat.values, k=1)
pairs = []
for i, j in zip(*upper):
    pairs.append({
        "pair": f"{all_markers[i]} — {all_markers[j]}",
        "co_6h": co_by_time["6h"].iloc[i, j],
        "co_48h": co_by_time["48h"].iloc[i, j],
        "log2_FC": lfc_mat.iloc[i, j],
    })
pairs_df = pd.DataFrame(pairs)

top_up = pairs_df.nlargest(25, "log2_FC").sort_values("log2_FC")
top_down = pairs_df.nsmallest(25, "log2_FC").sort_values("log2_FC", ascending=False)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 10))

# Top 25 increased co-assignment at 48h
ax1.barh(range(len(top_up)), top_up["log2_FC"].values, color="#d6604d")
ax1.set_yticks(range(len(top_up)))
ax1.set_yticklabels(top_up["pair"].values, fontsize=7)
ax1.set_xlabel("log2 FC (48h / 6h)")
ax1.set_title("Top 25 INCREASED co-assignment (48h vs 6h)", fontsize=11)
# Annotate with actual probabilities
for idx, (_, row) in enumerate(top_up.iterrows()):
    ax1.annotate(f"  6h={row['co_6h']:.2f} → 48h={row['co_48h']:.2f}",
                 xy=(max(0, row['log2_FC']), idx), fontsize=5.5, va="center", color="grey")

# Top 25 decreased co-assignment at 48h
ax2.barh(range(len(top_down)), top_down["log2_FC"].values, color="#4393c3")
ax2.set_yticks(range(len(top_down)))
ax2.set_yticklabels(top_down["pair"].values, fontsize=7)
ax2.set_xlabel("log2 FC (48h / 6h)")
ax2.set_title("Top 25 DECREASED co-assignment (48h vs 6h)", fontsize=11)
for idx, (_, row) in enumerate(top_down.iterrows()):
    ax2.annotate(f"  6h={row['co_6h']:.2f} → 48h={row['co_48h']:.2f}",
                 xy=(min(0, row['log2_FC']), idx), fontsize=5.5, va="center", color="grey", ha="right")

fig.suptitle("NALM-6 + healthy T | Blinatumomab CD8: co-assignment changes (48h vs 6h)",
             fontsize=13, y=1.01)
plt.tight_layout()
plt.show()

# Summary stats
print(f"\nTotal pairs: {len(pairs_df)}")
print(f"Pairs with |log2 FC| > 1: {(pairs_df['log2_FC'].abs() > 1).sum()}")
print(f"Pairs with |log2 FC| > 2: {(pairs_df['log2_FC'].abs() > 2).sum()}")
print(f"\nTop 5 increased:")
print(pairs_df.nlargest(5, "log2_FC")[["pair", "co_6h", "co_48h", "log2_FC"]].to_string(index=False))
print(f"\nTop 5 decreased:")
print(pairs_df.nsmallest(5, "log2_FC")[["pair", "co_6h", "co_48h", "log2_FC"]].to_string(index=False))

In [ ]:
# =============================================================================
# Per-marker co-assignment profile stability
# NALM-6 + healthy T, Blinatumomab CD8 — 6h vs 48h
#
# For each marker, its "co-assignment profile" = vector of co-assignment
# probabilities with every other marker. Compare profiles between 6h and 48h
# using cosine similarity. Low similarity = marker reorganized its neighborhood.
# =============================================================================
from scipy.spatial.distance import cosine

# co_by_time was computed in the previous cell (6h and 48h co-assignment matrices)
# Each is a marker × marker DataFrame of co-assignment probabilities

co_6h = co_by_time["6h"]
co_48h = co_by_time["48h"]

# --- Per-marker profile similarity (cosine) between 6h and 48h ---
profile_sim = {}
for marker in all_markers:
    # Co-assignment profile = row of co-assignment matrix (excluding self)
    others = [m for m in all_markers if m != marker]
    vec_6h = co_6h.loc[marker, others].values
    vec_48h = co_48h.loc[marker, others].values
    # Cosine similarity (1 - cosine distance)
    profile_sim[marker] = 1 - cosine(vec_6h, vec_48h)

sim_df = pd.DataFrame({
    "marker": list(profile_sim.keys()),
    "cosine_sim": list(profile_sim.values()),
}).sort_values("cosine_sim")

print(f"Profile similarity (cosine) between 6h and 48h co-assignment profiles:")
print(f"  Mean: {sim_df['cosine_sim'].mean():.3f}")
print(f"  Min:  {sim_df['cosine_sim'].min():.3f} ({sim_df.iloc[0]['marker']})")
print(f"  Max:  {sim_df['cosine_sim'].max():.3f} ({sim_df.iloc[-1]['marker']})")

# --- Figure 1: Bar chart — all markers sorted by profile stability ---
fig, ax = plt.subplots(figsize=(8, max(14, len(sim_df) * 0.12)))
colors = ["#d6604d" if v < 0.9 else "#f4a582" if v < 0.95 else "#92c5de" if v < 0.99 else "#4393c3"
          for v in sim_df["cosine_sim"].values]
ax.barh(range(len(sim_df)), sim_df["cosine_sim"].values, color=colors)
ax.set_yticks(range(len(sim_df)))
ax.set_yticklabels(sim_df["marker"].values, fontsize=5)
ax.set_xlabel("Cosine similarity (6h vs 48h co-assignment profile)")
ax.set_title("NALM-6 + healthy T | Blinatumomab CD8\n"
             "Per-marker co-assignment profile stability (6h → 48h)", fontsize=11)
ax.axvline(0.95, ls="--", color="grey", lw=0.8, alpha=0.6, label="0.95 threshold")
ax.legend(fontsize=8)
plt.tight_layout()
plt.show()

# --- Figure 2: Top 20 most reorganized markers — what changed? ---
top_changed = sim_df.head(20).copy()

# For each changed marker, find which co-assignment partners changed most
fig, axes = plt.subplots(4, 5, figsize=(24, 20))
axes_flat = axes.flatten()

for idx, (_, row) in enumerate(top_changed.iterrows()):
    if idx >= 20:
        break
    ax = axes_flat[idx]
    marker = row["marker"]
    others = [m for m in all_markers if m != marker]

    delta = co_48h.loc[marker, others] - co_6h.loc[marker, others]
    delta_sorted = delta.sort_values()

    # Show top 5 decreased and top 5 increased
    show = pd.concat([delta_sorted.head(5), delta_sorted.tail(5)])
    colors_bar = ["#4393c3" if v < 0 else "#d6604d" for v in show.values]
    ax.barh(range(len(show)), show.values, color=colors_bar)
    ax.set_yticks(range(len(show)))
    ax.set_yticklabels(show.index, fontsize=6)
    ax.axvline(0, color="k", linewidth=0.5)
    ax.set_title(f"{marker}\n(sim={row['cosine_sim']:.3f})", fontsize=8, fontweight="bold")
    ax.tick_params(axis="x", labelsize=6)
    if idx % 5 == 0:
        ax.set_ylabel("Δ co-assignment (48h − 6h)", fontsize=6)

fig.suptitle("NALM-6 + healthy T | Blinatumomab CD8: top 20 most reorganized markers\n"
             "Top 5 gained & lost co-assignment partners (48h vs 6h)",
             fontsize=13, y=1.01)
plt.tight_layout()
plt.show()

# --- Figure 3: Heatmap of Δ co-assignment for the top 20 changed markers ---
top_markers = top_changed["marker"].tolist()
delta_mat = co_48h.loc[top_markers, all_markers] - co_6h.loc[top_markers, all_markers]

# Only show columns (partners) with at least one large change
max_delta_per_partner = delta_mat.abs().max(axis=0)
notable_partners = max_delta_per_partner[max_delta_per_partner > 0.1].index.tolist()
# Always include the top markers themselves
notable_partners = sorted(set(notable_partners) | set(top_markers))
delta_show = delta_mat[notable_partners]

vmax = np.percentile(np.abs(delta_show.values), 98)

fig, ax = plt.subplots(figsize=(max(10, len(notable_partners) * 0.15),
                                 max(6, len(top_markers) * 0.3)))
sns.heatmap(delta_show, cmap="RdBu_r", center=0, vmin=-vmax, vmax=vmax, ax=ax,
            xticklabels=True, yticklabels=True, linewidths=0.3,
            cbar_kws={"label": "Δ co-assignment (48h − 6h)", "shrink": 0.6})
ax.set_title("Co-assignment change for top 20 reorganized markers", fontsize=11)
ax.tick_params(axis="y", labelsize=7)
ax.tick_params(axis="x", labelsize=5, rotation=90)
plt.tight_layout()
plt.show()

In [ ]:
# =============================================================================
# Interactive marker explorer: co-community neighbors across conditions
# =============================================================================

def plot_marker_neighbors(marker, percell_df, adata_obs, all_markers, top_n=20):
    """
    For a given marker, compute how often each other marker shares its community
    across cells, and compare:
      - Row 1: 6h vs 48h  |  Mock vs Blinatumomab
      - Row 2: 4 cell systems side by side
    """
    df = percell_df.copy()
    cidx = df.index.intersection(adata_obs.index)
    df = df.loc[cidx]
    df["condition"] = adata_obs.loc[cidx, "condition"].values
    df["time"] = adata_obs.loc[cidx, "time"].values
    df["system"] = adata_obs.loc[cidx, "system"].values

    other_markers = [m for m in all_markers if m != marker]

    def neighbor_freq(sub_df):
        """Fraction of cells where each other marker is in the same community as `marker`."""
        ref = sub_df[marker].values
        freqs = {}
        for m in other_markers:
            freqs[m] = (sub_df[m].values == ref).mean()
        return pd.Series(freqs).sort_values(ascending=False)

    # Compute frequencies per group
    freq_6h = neighbor_freq(df[df["time"] == "6h"])
    freq_48h = neighbor_freq(df[df["time"] == "48h"])
    freq_mock = neighbor_freq(df[df["condition"] == "Mock"])
    freq_blina = neighbor_freq(df[df["condition"] == "Blinatumomab"])

    systems = sorted(df["system"].unique())
    freq_sys = {s: neighbor_freq(df[df["system"] == s]) for s in systems}

    # Union of top_n neighbors across all groups
    top_set = set()
    for freq in [freq_6h, freq_48h, freq_mock, freq_blina] + list(freq_sys.values()):
        top_set |= set(freq.head(top_n).index)
    top_markers = sorted(top_set, key=lambda m: -(freq_6h.get(m, 0) + freq_48h.get(m, 0)) / 2)

    # --- Row 1: Time comparison + Treatment comparison ---
    fig, axes = plt.subplots(1, 2, figsize=(16, max(6, len(top_markers) * 0.28)))

    # Panel A: 6h vs 48h
    ax = axes[0]
    y = np.arange(len(top_markers))
    w = 0.35
    vals_6h = [freq_6h.get(m, 0) for m in top_markers]
    vals_48h = [freq_48h.get(m, 0) for m in top_markers]
    ax.barh(y - w/2, vals_6h, w, label="6h", color=TIME_PALETTE["6h"], edgecolor="white", linewidth=0.3)
    ax.barh(y + w/2, vals_48h, w, label="48h", color=TIME_PALETTE["48h"], edgecolor="white", linewidth=0.3)
    ax.set_yticks(y)
    ax.set_yticklabels(top_markers, fontsize=7)
    ax.set_xlabel("Co-community frequency")
    ax.set_title(f"6h vs 48h", fontsize=11)
    ax.legend(fontsize=9)
    ax.invert_yaxis()
    ax.set_xlim(0, 1)

    # Panel B: Mock vs Blinatumomab
    ax = axes[1]
    vals_mock = [freq_mock.get(m, 0) for m in top_markers]
    vals_blina = [freq_blina.get(m, 0) for m in top_markers]
    ax.barh(y - w/2, vals_mock, w, label="Mock", color=COND_PALETTE["Mock"], edgecolor="white", linewidth=0.3)
    ax.barh(y + w/2, vals_blina, w, label="Blinatumomab", color=COND_PALETTE["Blinatumomab"], edgecolor="white", linewidth=0.3)
    ax.set_yticks(y)
    ax.set_yticklabels(top_markers, fontsize=7)
    ax.set_xlabel("Co-community frequency")
    ax.set_title(f"Mock vs Blinatumomab", fontsize=11)
    ax.legend(fontsize=9)
    ax.invert_yaxis()
    ax.set_xlim(0, 1)

    fig.suptitle(f"{marker}: co-community neighbors", fontsize=13, y=1.02)
    plt.tight_layout()
    plt.show()

    # --- Row 2: 4 cell systems ---
    n_sys = len(systems)
    fig, axes = plt.subplots(1, n_sys, figsize=(5 * n_sys, max(6, len(top_markers) * 0.28)),
                              sharey=True)
    sys_colors = ["#4878d0", "#ee854a", "#6acc65", "#d65f5f"]
    for idx, (sys_name, ax) in enumerate(zip(systems, axes)):
        vals = [freq_sys[sys_name].get(m, 0) for m in top_markers]
        ax.barh(y, vals, color=sys_colors[idx % len(sys_colors)], edgecolor="white", linewidth=0.3)
        ax.set_title(sys_name, fontsize=9)
        ax.set_xlim(0, 1)
        ax.invert_yaxis()
        if idx == 0:
            ax.set_yticks(y)
            ax.set_yticklabels(top_markers, fontsize=7)
        ax.set_xlabel("Co-community freq")

    fig.suptitle(f"{marker}: co-community neighbors by cell system", fontsize=13, y=1.02)
    plt.tight_layout()
    plt.show()

    # --- Delta plot: what changes most? ---
    delta_time = pd.Series({m: freq_48h.get(m, 0) - freq_6h.get(m, 0) for m in top_markers})
    delta_cond = pd.Series({m: freq_blina.get(m, 0) - freq_mock.get(m, 0) for m in top_markers})

    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, max(5, len(top_markers) * 0.25)))

    order_t = delta_time.sort_values()
    colors_t = ["#c44e52" if v > 0 else "#55a868" for v in order_t.values]
    ax1.barh(range(len(order_t)), order_t.values, color=colors_t)
    ax1.set_yticks(range(len(order_t)))
    ax1.set_yticklabels(order_t.index, fontsize=7)
    ax1.axvline(0, color="k", linewidth=0.5)
    ax1.set_xlabel("Δ co-community freq (48h − 6h)")
    ax1.set_title("Time effect")

    order_c = delta_cond.sort_values()
    colors_c = ["#dd8452" if v > 0 else "#4c72b0" for v in order_c.values]
    ax2.barh(range(len(order_c)), order_c.values, color=colors_c)
    ax2.set_yticks(range(len(order_c)))
    ax2.set_yticklabels(order_c.index, fontsize=7)
    ax2.axvline(0, color="k", linewidth=0.5)
    ax2.set_xlabel("Δ co-community freq (Blina − Mock)")
    ax2.set_title("Treatment effect")

    fig.suptitle(f"{marker}: change in co-community neighbors", fontsize=13, y=1.02)
    plt.tight_layout()
    plt.show()

# --- Run for a marker ---
QUERY_MARKER = "CD3e"  # <-- change this to explore different markers
plot_marker_neighbors(QUERY_MARKER, percell["Leiden CPM"], adata.obs, all_markers, top_n=20)